# 05 — Test Module 3: Super-graph Construction

Verify that super-graph is built correctly:
- Off-diagonal binary (any edge between clusters → 1)
- Self-loop binary from ORIGINAL graph (Quy ước 2, Cách α)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os
PROJECT_ROOT = '/content/drive/MyDrive/Project_GraphML/ms-zerogad'
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

In [ ]:
import torch
from ms_zerogad.data.loader import load_graph_dataset
from ms_zerogad.data.preprocessing import sparse_to_torch_dense, feature_to_torch
from ms_zerogad.modules.unification import GlobalUnification
from ms_zerogad.modules.clustering import RFFClustering
from ms_zerogad.modules.supergraph import build_super_graph
from ms_zerogad.utils.tracking import MembershipTracker

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Build super-graph from Cora

In [ ]:
A_sp, X_sp, y_np = load_graph_dataset('/content/drive/MyDrive/Project_GraphML/ms-zerogad/ms_zerogad/data/raw/Cora.mat')
A = sparse_to_torch_dense(A_sp).to(device)
X = feature_to_torch(X_sp, dense=True).to(device)

module1 = GlobalUnification(d_prime=8).to(device)
clustering = RFFClustering().to(device)

X_unified = module1(X, A)
n = X_unified.shape[0]

# Cluster
tracker = MembershipTracker(n_original=n)
P = clustering(X_unified, A, num_clusters=n // 2)
membership = tracker.update(P)
K = P.shape[1]

# Build super-graph
X_super, A_super_hat = build_super_graph(
    X=X_unified,
    A=A,
    P=P,
    membership_for_self_loop=membership,
    A_original=A,
)

print(f'X_super: {X_super.shape}')
print(f'A_super_hat: {A_super_hat.shape}')

## Verify super-adjacency properties

In [ ]:
A_raw = (A_super_hat - torch.eye(K, device=device)).detach()  # remove the +I we added

# Off-diagonal
diag_mask = torch.eye(K, device=device, dtype=torch.bool)
offdiag = A_raw.masked_fill(diag_mask, 0)
diag = A_raw.diagonal()

# Off-diag should be binary
is_offdiag_binary = ((offdiag == 0) | (offdiag == 1)).all().item()
# Diag should be binary
is_diag_binary = ((diag == 0) | (diag == 1)).all().item()
# Symmetric
is_symmetric = torch.allclose(offdiag, offdiag.T, atol=1e-6)

print(f'Off-diagonal binary: {is_offdiag_binary}')
print(f'Self-loop binary:    {is_diag_binary}')
print(f'Symmetric:           {is_symmetric}')
print()
print(f'Off-diagonal nnz:  {int(offdiag.sum().item())} (out of {K * (K-1)} possible)')
print(f'Self-loops set:    {int(diag.sum().item())} / {K} clusters')

## Verify self-loop matches original-graph internal edges

In [ ]:
A_cpu = A.cpu()

errors = 0
for j in range(K):
    nodes = membership[j]
    if len(nodes) < 2:
        # No internal edge possible
        expected_self_loop = 0
    else:
        idx = torch.tensor(nodes)
        sub_A = A_cpu[idx][:, idx]
        has_internal_edge = torch.triu(sub_A, diagonal=1).sum() > 0
        expected_self_loop = 1 if has_internal_edge else 0

    actual = int(diag[j].item())
    if actual != expected_self_loop:
        errors += 1

print(f'Self-loop check: {K - errors}/{K} correct (errors={errors})')